# Name That LLM — Qwen LoRA + TF-IDF (GPU)

Two-stage LoRA on **Qwen3-8B-Base** (4-bit), then blend with the CPU TF-IDF model.

**Runtime settings (required)**

- Internet: **Off**
- Accelerator: **GPU T4 x2**
- Persistence: files only
- Time limit: 9 hours (training + ~4000-row rerun)

**Inputs to attach**

1. Competition data: `name-that-llm`
2. Kaggle Model: `qwen-lm/qwen-3` → Transformers → **8b-base** / 1  
   (Apache 2.0; path usually `/kaggle/input/qwen-3/transformers/8b-base/1`)
3. Extra public passages (at least one; more is better):
   - `nishantdahal/ntl-two-stage-lora` — use **`pool_v1.parquet` only** (CC BY-SA 4.0 public pool). Adapters in that dataset are ignored.
   - `prvsiyan/prvsiyans-extended-name-that-llm-dataset`
   - `seyitkaangunes/ntl-extra-api-passages`
   - `willrice/name-that-llm-openrouter-harvest`

Phone verification is required on Kaggle before GPU sessions.

Do not copy other teams' LoRA checkpoints. This notebook trains its own adapters on the attached public text.


In [ ]:
"""Competition labels, families, and closed-class word lists."""

from __future__ import annotations

LABELS: tuple[str, ...] = (
    "gpt-5.5",
    "gpt-5.6-luna",
    "gpt-5.6-sol",
    "gpt-5.6-terra",
    "gpt-6-astra",
    "gemini-3.1-pro-preview",
    "gemini-3.6-flash",
    "gemini-3.7-flash",
    "gemini-3.8-flash",
    "claude-fable-5",
    "claude-opus-4-6",
    "claude-opus-4-8",
    "claude-opus-5",
    "claude-sonnet-5",
    "qwen3.8-max",
    "glm-5.3",
    "glm-5.3-flash",
    "kimi-k3",
)

FAMILY_OF: dict[str, str] = {}
for _name in LABELS:
    if _name.startswith("gpt-"):
        FAMILY_OF[_name] = "openai"
    elif _name.startswith("gemini-"):
        FAMILY_OF[_name] = "google"
    elif _name.startswith("claude-"):
        FAMILY_OF[_name] = "anthropic"
    elif _name.startswith("qwen"):
        FAMILY_OF[_name] = "qwen"
    elif _name.startswith("glm-"):
        FAMILY_OF[_name] = "zai"
    elif _name.startswith("kimi-"):
        FAMILY_OF[_name] = "moonshot"
    else:
        FAMILY_OF[_name] = "other"

FAMILIES: tuple[str, ...] = (
    "openai",
    "google",
    "anthropic",
    "qwen",
    "zai",
    "moonshot",
)

# Marks that some labs almost never produce in the host training set.
# Used as soft probability penalties, never as hard bans.
MARK_PATTERNS: dict[str, str] = {
    "emdash": r"—",
    "endash": r"–",
    "curly_dq": r"[“”]",
    "curly_sq": r"[‘’]",
    "ellipsis_uni": r"…",
    "excl": r"!",
    "contraction": r"(?i)\b(?:['’]t|['’]s|['’]re|['’]ve|['’]ll|['’]d|n't)\b",
}

FUNCTION_WORDS: frozenset[str] = frozenset(
    """
    a an the and or but if while although though because as at by for from in into
    of on to with without within through during before after above below between
    among against about across around until unless whether
    this that these those i you he she it we they me him her us them my your his
    its our their mine yours hers ours theirs
    is are was were be been being do does did have has had can could may might
    must shall should will would
    not no nor so than then there here when where what which who whom whose how
    why whom
    also just only even still yet already often never always perhaps maybe
    however therefore thus hence moreover furthermore instead rather quite
    very much many more most less least such own same other another each every
    both few several some any all
    """.split()
)

STYLE_MARKERS: tuple[str, ...] = (
    "however",
    "therefore",
    "thus",
    "hence",
    "moreover",
    "furthermore",
    "notably",
    "indeed",
    "perhaps",
    "simply",
    "essentially",
    "fundamentally",
    "delve",
    "tapestry",
    "underscore",
    "crucial",
    "vital",
    "intricate",
    "landscape",
    "nuanced",
    "robust",
    "leverage",
    "not only",
    "in other words",
    "that is",
    "for instance",
    "such as",
    "in contrast",
    "on the other hand",
    "as a result",
    "in fact",
    "at its core",
    "the key",
)

KAGGLE_TRAIN_CANDIDATES: tuple[str, ...] = (
    "/kaggle/input/competitions/name-that-llm/train.csv",
    "/kaggle/input/name-that-llm/train.csv",
)

KAGGLE_TEST_CANDIDATES: tuple[str, ...] = (
    "/kaggle/input/competitions/name-that-llm/test.csv",
    "/kaggle/input/name-that-llm/test.csv",
)


In [ ]:
"""Stylometric dense features and text views for n-gram vectorizers."""

from __future__ import annotations

import re
from typing import Iterable

import numpy as np


_WORD_RE = re.compile(r"[A-Za-z]+(?:['’][A-Za-z]+)?|\d+")
_SENT_RE = re.compile(r"[.!?]+")
_OXFORD_RE = re.compile(r",\s+(?:and|or)\s+", re.I)
_HYPHEN_WORD_RE = re.compile(r"\b\w+-\w+\b")
_LY_RE = re.compile(r"\b\w+ly\b", re.I)
_DIGIT_RE = re.compile(r"\d")
_UPPER_RE = re.compile(r"[A-Z]")
_LETTER_RE = re.compile(r"[A-Za-z]")
_CONTRACTION_RE = re.compile(
    r"(?i)\b\w+(?:n't|['’]t|['’]s|['’]re|['’]ve|['’]ll|['’]d)\b"
)
_START_RES = {
    "start_the": re.compile(r"^The\b"),
    "start_in": re.compile(r"^In\b"),
    "start_imagine": re.compile(r"^Imagine\b"),
    "start_when": re.compile(r"^When\b"),
    "start_this": re.compile(r"^This\b"),
    "start_a": re.compile(r"^A\b"),
    "start_as": re.compile(r"^As\b"),
}
_PRONOUN_RES = {
    "you": re.compile(r"\byou\b", re.I),
    "we": re.compile(r"\bwe\b", re.I),
    "it": re.compile(r"\bit\b", re.I),
    "one": re.compile(r"\bone\b", re.I),
}
_MARKER_RES = tuple((m, re.compile(re.escape(m), re.I)) for m in STYLE_MARKERS)
_MARK_RES = {name: re.compile(pat) for name, pat in MARK_PATTERNS.items()}


def to_word_shape(text: str) -> str:
    parts: list[str] = []
    for tok in re.findall(r"\S+", text):
        chars: list[str] = []
        for ch in tok:
            if ch.isupper():
                chars.append("X")
            elif ch.islower():
                chars.append("x")
            elif ch.isdigit():
                chars.append("d")
            else:
                chars.append(ch)
        shape = "".join(chars)
        shape = re.sub(r"x{4,}", "x+", shape)
        shape = re.sub(r"X{4,}", "X+", shape)
        shape = re.sub(r"d{3,}", "d+", shape)
        parts.append(shape)
    return " ".join(parts)


def to_func_view(text: str) -> str:
    toks = re.findall(r"[A-Za-z]+|[^\w\s]", text)
    out: list[str] = []
    for tok in toks:
        low = tok.lower()
        if low in FUNCTION_WORDS:
            out.append(low)
        elif tok.isalnum():
            out.append("CONTENT")
        else:
            out.append(tok)
    return " ".join(out)


def to_punct_view(text: str) -> str:
    chars: list[str] = []
    prev_space = False
    for ch in text:
        if ch.isalnum():
            continue
        if ch.isspace():
            if not prev_space:
                chars.append(" ")
            prev_space = True
            continue
        chars.append(ch)
        prev_space = False
    return "".join(chars).strip()


def _safe_div(a: float, b: float) -> float:
    return float(a) / float(b) if b else 0.0


def dense_feature_names() -> list[str]:
    names = [
        "n_chars",
        "n_words",
        "n_sents",
        "avg_word_len",
        "std_word_len",
        "avg_sent_len",
        "ttr",
        "long_word_rate",
        "short_word_rate",
        "comma_rate",
        "semi_rate",
        "colon_rate",
        "emdash_rate",
        "endash_rate",
        "hyphen_rate",
        "excl_rate",
        "qmark_rate",
        "paren_rate",
        "curly_q_rate",
        "straight_q_rate",
        "apos_curly_rate",
        "ellipsis_rate",
        "slash_rate",
        "oxford_rate",
        "hyphen_word_rate",
        "contraction_rate",
        "ly_rate",
        "digit_rate",
        "upper_rate",
        "comma_per_sent",
        "unique_punct",
        "ends_period",
        "has_emdash",
        "has_curly",
        "has_contraction",
    ]
    names.extend(_START_RES.keys())
    names.extend(f"pron_{k}" for k in _PRONOUN_RES)
    names.extend(f"mark_{m.replace(' ', '_')}" for m, _ in _MARKER_RES)
    names.extend(f"has_{name}" for name in MARK_PATTERNS)
    return names


def extract_dense_row(text: str) -> list[float]:
    n_chars = max(len(text), 1)
    words = _WORD_RE.findall(text)
    n_words = len(words)
    sents = [s for s in _SENT_RE.split(text) if s.strip()]
    n_sents = max(len(sents), 1)
    lengths = [len(w) for w in words] or [0]
    avg_w = float(np.mean(lengths))
    std_w = float(np.std(lengths))
    lower_words = [w.lower() for w in words]
    ttr = _safe_div(len(set(lower_words)), n_words)
    long_rate = _safe_div(sum(1 for L in lengths if L >= 10), n_words)
    short_rate = _safe_div(sum(1 for L in lengths if L <= 3), n_words)

    def rate(count: int) -> float:
        return 1000.0 * count / n_chars

    row = [
        float(n_chars),
        float(n_words),
        float(n_sents),
        avg_w,
        std_w,
        _safe_div(n_words, n_sents),
        ttr,
        long_rate,
        short_rate,
        rate(text.count(",")),
        rate(text.count(";")),
        rate(text.count(":")),
        rate(text.count("—")),
        rate(text.count("–")),
        rate(text.count("-")),
        rate(text.count("!")),
        rate(text.count("?")),
        rate(text.count("(") + text.count(")")),
        rate(len(re.findall(r"[“”‘’]", text))),
        rate(text.count('"') + text.count("'")),
        rate(text.count("’")),
        rate(text.count("…") + text.count("...")),
        rate(text.count("/") + text.count("\\")),
        rate(len(_OXFORD_RE.findall(text))),
        _safe_div(len(_HYPHEN_WORD_RE.findall(text)), n_words),
        _safe_div(len(_CONTRACTION_RE.findall(text)), n_words),
        _safe_div(len(_LY_RE.findall(text)), n_words),
        rate(len(_DIGIT_RE.findall(text))),
        _safe_div(len(_UPPER_RE.findall(text)), max(len(_LETTER_RE.findall(text)), 1)),
        _safe_div(text.count(","), n_sents),
        float(len(set(ch for ch in text if not ch.isalnum() and not ch.isspace()))),
        1.0 if text.rstrip().endswith(".") else 0.0,
        1.0 if "—" in text else 0.0,
        1.0 if re.search(r"[“”‘’]", text) else 0.0,
        1.0 if _CONTRACTION_RE.search(text) else 0.0,
    ]
    for rx in _START_RES.values():
        row.append(1.0 if rx.match(text) else 0.0)
    for rx in _PRONOUN_RES.values():
        row.append(_safe_div(len(rx.findall(text)), n_words))
    for _, rx in _MARKER_RES:
        row.append(float(len(rx.findall(text))))
    for rx in _MARK_RES.values():
        row.append(1.0 if rx.search(text) else 0.0)
    return row


def extract_dense(texts: Iterable[str]) -> np.ndarray:
    return np.asarray([extract_dense_row(t) for t in texts], dtype=np.float32)


def mark_hits(text: str) -> dict[str, bool]:
    return {name: bool(rx.search(text)) for name, rx in _MARK_RES.items()}


In [ ]:
"""Multi-view n-gram + stylometry authorship classifier."""

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable, Sequence

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

    FAMILIES,
    FAMILY_OF,
    KAGGLE_TEST_CANDIDATES,
    KAGGLE_TRAIN_CANDIDATES,
    LABELS,
    MARK_PATTERNS,
)
    extract_dense,
    mark_hits,
    to_word_shape,
)

WORD_TOKEN_PATTERN = r"(?u)\b\w+\b|[^\w\s]"


def balanced_accuracy(y_true: Sequence[str], y_pred: Sequence[str]) -> float:
    return float(balanced_accuracy_score(y_true, y_pred))


def softmax_scores(scores: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    z = np.asarray(scores, dtype=np.float64) / max(temperature, 1e-6)
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.clip(e.sum(axis=1, keepdims=True), 1e-12, None)


def rebalance_probs(probs: np.ndarray, target: np.ndarray | None = None, n_iter: int = 8) -> np.ndarray:
    q = np.clip(np.asarray(probs, dtype=np.float64), 1e-12, None)
    k = q.shape[1]
    if target is None:
        target = np.full(k, 1.0 / k)
    target = np.asarray(target, dtype=np.float64)
    target = target / target.sum()
    for _ in range(n_iter):
        q = q / q.sum(axis=1, keepdims=True)
        col = q.mean(axis=0)
        q = q * (target / np.clip(col, 1e-12, None))
    return q / q.sum(axis=1, keepdims=True)


def _first_existing(paths: Iterable[str | Path]) -> Path | None:
    for p in paths:
        path = Path(p)
        if path.exists():
            return path
    return None


def find_train_path(extra: Sequence[str | Path] | None = None) -> Path:
    candidates = list(extra or [])
    candidates.extend(KAGGLE_TRAIN_CANDIDATES)
    found = _first_existing(candidates)
    if found is None:
        raise FileNotFoundError("train.csv not found")
    return found


def find_test_path(extra: Sequence[str | Path] | None = None) -> Path:
    candidates = list(extra or [])
    candidates.extend(KAGGLE_TEST_CANDIDATES)
    found = _first_existing(candidates)
    if found is None:
        raise FileNotFoundError("test.csv not found")
    return found


def load_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)


def normalize_frame(df: pd.DataFrame) -> pd.DataFrame:
    cols = {c.lower(): c for c in df.columns}
    out = pd.DataFrame()
    if "id" in cols:
        out["id"] = df[cols["id"]].astype(str)
    else:
        out["id"] = [f"row{i:06d}" for i in range(len(df))]
    if "text" not in cols:
        raise ValueError(f"missing text column in {list(df.columns)}")
    out["text"] = df[cols["text"]].fillna("").astype(str)
    if "model" in cols:
        out["model"] = df[cols["model"]].astype(str)
    if "prompt_id" in cols:
        out["prompt_id"] = df[cols["prompt_id"]].astype(str)
    else:
        out["prompt_id"] = out["id"]
    return out


def load_extra_frames(roots: Sequence[str | Path]) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    skip_names = {"train.csv", "test.csv", "sample_submission.parquet", "sample_submission.csv"}
    for root in roots:
        root_path = Path(root)
        if not root_path.exists():
            continue
        paths = [root_path] if root_path.is_file() else list(root_path.rglob("*"))
        for path in paths:
            if not path.is_file() or path.suffix.lower() not in {".csv", ".parquet"}:
                continue
            if path.name in skip_names:
                continue
            low = str(path).lower()
            if path.suffix.lower() in {".safetensors", ".bin", ".pt", ".pth", ".ckpt", ".gguf", ".npz"}:
                continue
            if any(m in low for m in ("/adapter/", "lora8b", "lora14b")):
                continue
            try:
                if path.stat().st_size > 120_000_000:
                    continue
            except OSError:
                continue
            try:
                df = normalize_frame(load_table(path))
            except Exception:
                continue
            if "model" not in df.columns:
                continue
            df = df[df["model"].isin(LABELS) & df["text"].str.len().gt(40)]
            if df.empty:
                continue
            frames.append(df[["id", "prompt_id", "text", "model"]])
    if not frames:
        return pd.DataFrame(columns=["id", "prompt_id", "text", "model"])
    extra = pd.concat(frames, ignore_index=True)
    extra = extra.drop_duplicates(subset=["text"])
    return extra.reset_index(drop=True)


def make_vectorizers() -> list[tuple[str, TfidfVectorizer, Any]]:
    # Character unigrams keep em dashes, quotes and other single-mark fingerprints.
    return [
        (
            "char",
            TfidfVectorizer(
                analyzer="char",
                ngram_range=(1, 5),
                min_df=2,
                max_features=300_000,
                sublinear_tf=True,
                lowercase=False,
            ),
            None,
        ),
        (
            "word",
            TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 2),
                min_df=2,
                max_features=200_000,
                sublinear_tf=True,
                lowercase=False,
                token_pattern=WORD_TOKEN_PATTERN,
            ),
            None,
        ),
        (
            "shape",
            TfidfVectorizer(
                analyzer="word",
                ngram_range=(2, 4),
                min_df=3,
                max_features=20_000,
                sublinear_tf=True,
                lowercase=False,
                token_pattern=r"\S+",
            ),
            to_word_shape,
        ),
    ]


def _transform_view(vec: TfidfVectorizer, texts: Sequence[str], preprocessor, fit: bool) -> csr_matrix:
    series = pd.Series(list(texts), dtype=str)
    if preprocessor is not None:
        series = series.map(preprocessor)
    if fit:
        return vec.fit_transform(series)
    return vec.transform(series)


def fit_mark_illegal(texts: Sequence[str], labels: Sequence[str]) -> dict[str, set[str]]:
    illegal: dict[str, set[str]] = {}
    frame = pd.DataFrame({"text": list(texts), "model": list(labels)})
    for name, pat in MARK_PATTERNS.items():
        rates = frame.groupby("model")["text"].apply(lambda s: s.str.contains(pat, regex=True).mean())
        illegal[name] = set(rates[rates <= 1e-12].index.tolist())
    return illegal


def apply_mark_penalties(
    probs: np.ndarray,
    texts: Sequence[str],
    classes: Sequence[str],
    illegal: dict[str, set[str]],
    factor: float = 0.08,
) -> np.ndarray:
    out = np.asarray(probs, dtype=np.float64).copy()
    class_index = {c: i for i, c in enumerate(classes)}
    for i, text in enumerate(texts):
        hits = mark_hits(text)
        for mark, present in hits.items():
            if not present:
                continue
            for lab in illegal.get(mark, ()):
                j = class_index.get(lab)
                if j is not None:
                    out[i, j] *= factor
        s = out[i].sum()
        if s > 0:
            out[i] /= s
    return out


def _logit() -> LogisticRegression:
    return LogisticRegression(
        C=10.0,
        max_iter=2500,
        class_weight="balanced",
        solver="lbfgs",
    )


def _map_proba(clf, x, all_names: Sequence[str]) -> np.ndarray:
    raw = clf.predict_proba(x)
    out = np.zeros((x.shape[0], len(all_names)), dtype=np.float64)
    idx = {str(c): i for i, c in enumerate(all_names)}
    for col, name in enumerate(clf.classes_):
        j = idx.get(str(name))
        if j is not None:
            out[:, j] = raw[:, col]
    s = out.sum(axis=1, keepdims=True)
    return np.divide(out, s, out=np.full_like(out, 1.0 / max(len(all_names), 1)), where=s > 0)


@dataclass
class NTLClassifier:
    extra_weight: float = 0.55
    blend_global: float = 1.0
    blend_hier: float = 0.0
    blend_style: float = 0.0
    rebalance: bool = False
    mark_factor: float = 0.25
    use_dense: bool = False
    classes_: np.ndarray = field(default_factory=lambda: np.array(LABELS))
    family_classes_: np.ndarray = field(default_factory=lambda: np.array(FAMILIES))

    def __post_init__(self) -> None:
        self.vectorizers_ = make_vectorizers()
        self.scaler_ = StandardScaler()
        self.global_clf_ = _logit()
        self.family_clf_ = _logit()
        self.within_: dict[str, LogisticRegression] = {}
        self.style_clf_ = None
        self.use_style_head_ = False
        self.illegal_marks_: dict[str, set[str]] = {}
        self.n_sparse_ = 0

    def _fit_sparse(self, texts: Sequence[str]) -> csr_matrix:
        blocks = []
        for _, vec, prep in self.vectorizers_:
            blocks.append(_transform_view(vec, texts, prep, fit=True))
        x = hstack(blocks).tocsr()
        self.n_sparse_ = x.shape[1]
        return x

    def _transform_sparse(self, texts: Sequence[str]) -> csr_matrix:
        blocks = []
        for _, vec, prep in self.vectorizers_:
            blocks.append(_transform_view(vec, texts, prep, fit=False))
        return hstack(blocks).tocsr()

    def _style_estimator(self, n_classes: int):
        try:
            import lightgbm as lgb

            return lgb.LGBMClassifier(
                n_estimators=240,
                learning_rate=0.05,
                num_leaves=31,
                subsample=0.9,
                colsample_bytree=0.8,
                min_child_samples=12,
                objective="multiclass",
                class_weight="balanced",
                verbosity=-1,
                n_jobs=-1,
            )
        except Exception:
            from sklearn.ensemble import HistGradientBoostingClassifier

            return HistGradientBoostingClassifier(
                max_depth=6,
                learning_rate=0.08,
                max_iter=180,
                class_weight="balanced",
            )

    def fit(
        self,
        texts: Sequence[str],
        y: Sequence[str],
        sample_weight: np.ndarray | None = None,
    ) -> "NTLClassifier":
        texts = [str(t) for t in texts]
        y = np.asarray(y)
        if sample_weight is None:
            sample_weight = np.ones(len(y), dtype=np.float64)
        present = set(y)
        self.classes_ = np.array([c for c in LABELS if c in present] + [c for c in present if c not in LABELS])
        fam = np.array([FAMILY_OF.get(v, "other") for v in y])

        x_sparse = self._fit_sparse(texts)
        if self.use_dense:
            x_dense = extract_dense(texts)
            x_dense_s = self.scaler_.fit_transform(x_dense)
            x_all = hstack([x_sparse, csr_matrix(x_dense_s)]).tocsr()
        else:
            x_all = x_sparse
            self.scaler_ = None

        self.global_clf_.fit(x_all, y, sample_weight=sample_weight)
        self.within_.clear()
        if self.blend_hier > 0:
            self.family_clf_.fit(x_all, fam, sample_weight=sample_weight)
            for family in FAMILIES:
                idx = np.where(fam == family)[0]
                labs = np.unique(y[idx])
                if len(labs) <= 1 or len(idx) < 8:
                    continue
                clf = _logit()
                clf.fit(x_all[idx], y[idx], sample_weight=sample_weight[idx])
                self.within_[family] = clf

        self.style_clf_ = None
        if self.blend_style > 0:
            self.style_clf_ = self._style_estimator(len(self.classes_))
            try:
                self.style_clf_.fit(x_dense_s, y, sample_weight=sample_weight)
            except TypeError:
                self.style_clf_.fit(x_dense_s, y)

        self.illegal_marks_ = fit_mark_illegal(texts, y)
        return self

    def predict_proba(self, texts: Sequence[str]) -> np.ndarray:
        texts = [str(t) for t in texts]
        x_sparse = self._transform_sparse(texts)
        if self.use_dense and self.scaler_ is not None:
            x_dense = extract_dense(texts)
            x_dense_s = self.scaler_.transform(x_dense)
            x_all = hstack([x_sparse, csr_matrix(x_dense_s)]).tocsr()
        else:
            x_all = x_sparse
            x_dense_s = None

        class_names = [str(c) for c in self.classes_]
        p_global = _map_proba(self.global_clf_, x_all, class_names)
        p_hier = p_global
        if self.blend_hier > 0:
            p_fam = _map_proba(self.family_clf_, x_all, list(FAMILIES))
            p_hier = np.zeros_like(p_global)
            remaining: dict[str, list[str]] = {fam: [] for fam in FAMILIES}
            for lab in class_names:
                fam = FAMILY_OF.get(lab)
                if fam in remaining:
                    remaining[fam].append(lab)
            for family, labs in remaining.items():
                if not labs:
                    continue
                cols = [class_names.index(lab) for lab in labs]
                fam_col = list(FAMILIES).index(family)
                fam_p = p_fam[:, fam_col]
                if family in self.within_ and len(labs) > 1:
                    within_p = _map_proba(self.within_[family], x_all, labs)
                else:
                    within_p = np.full((len(texts), len(labs)), 1.0 / max(len(labs), 1))
                p_hier[:, cols] = within_p * fam_p[:, None]
            p_hier = p_hier / np.clip(p_hier.sum(axis=1, keepdims=True), 1e-12, None)

        if self.style_clf_ is not None and x_dense_s is not None and hasattr(self.style_clf_, "predict_proba"):
            p_style = _map_proba(self.style_clf_, x_dense_s, class_names)
        else:
            p_style = p_global

        w_g, w_h, w_s = self.blend_global, self.blend_hier, self.blend_style
        wsum = w_g + w_h + w_s
        probs = (w_g * p_global + w_h * p_hier + w_s * p_style) / wsum
        probs = apply_mark_penalties(probs, texts, class_names, self.illegal_marks_, self.mark_factor)
        if self.rebalance:
            probs = rebalance_probs(probs)
        return probs

    def predict(self, texts: Sequence[str]) -> np.ndarray:
        probs = self.predict_proba(texts)
        idx = probs.argmax(axis=1)
        return np.asarray(self.classes_)[idx]


def grouped_cv(
    train: pd.DataFrame,
    extra: pd.DataFrame | None = None,
    n_splits: int = 5,
    extra_weight: float = 0.55,
    seed: int = 42,
) -> dict[str, Any]:
    y = train["model"].astype(str).to_numpy()
    groups = train["prompt_id"].astype(str).to_numpy()
    texts = train["text"].astype(str).to_numpy()
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_pred = np.empty(len(train), dtype=object)
    oof_prob = np.zeros((len(train), len(LABELS)), dtype=np.float64)
    fold_scores: list[float] = []
    extra = extra if extra is not None and len(extra) else pd.DataFrame()

    for fold, (trn, val) in enumerate(splitter.split(train, y, groups=groups), start=1):
        tr_texts = list(texts[trn])
        tr_y = list(y[trn])
        weights = [1.0] * len(tr_texts)
        if len(extra):
            tr_texts.extend(extra["text"].astype(str).tolist())
            tr_y.extend(extra["model"].astype(str).tolist())
            weights.extend([extra_weight] * len(extra))
        model = NTLClassifier(extra_weight=extra_weight)
        model.fit(tr_texts, tr_y, sample_weight=np.asarray(weights, dtype=np.float64))
        val_texts = list(texts[val])
        proba = model.predict_proba(val_texts)
        pred = np.asarray(model.classes_)[proba.argmax(axis=1)]
        oof_pred[val] = pred
        col = {str(c): i for i, c in enumerate(model.classes_)}
        aligned = np.zeros((len(val), len(LABELS)))
        for j, lab in enumerate(LABELS):
            if lab in col:
                aligned[:, j] = proba[:, col[lab]]
        oof_prob[val] = aligned
        score = balanced_accuracy(y[val], pred)
        fold_scores.append(score)
        print(f"fold {fold}: balanced_acc={score:.4f}", flush=True)

    overall = balanced_accuracy(y, oof_pred)
    per_class = {}
    for lab in LABELS:
        mask = y == lab
        per_class[lab] = float((oof_pred[mask] == lab).mean()) if mask.any() else 0.0
    return {
        "overall": overall,
        "fold_scores": fold_scores,
        "per_class": per_class,
        "oof_pred": oof_pred,
        "oof_prob": oof_prob,
        "n_train": int(len(train)),
        "n_extra": int(len(extra)),
    }


def write_submission(ids: Sequence[str], models: Sequence[str], path: str | Path) -> Path:
    path = Path(path)
    sub = pd.DataFrame({"id": list(ids), "model": list(models)})
    if sub["id"].duplicated().any():
        raise ValueError("duplicate ids in submission")
    if sub.isnull().any().any():
        raise ValueError("nulls in submission")
    bad = set(sub["model"]) - set(LABELS)
    if bad:
        raise ValueError(f"invalid labels: {sorted(bad)}")
    path.parent.mkdir(parents=True, exist_ok=True)
    sub.to_parquet(path, index=False)
    return path


def make_synthetic(n_prompts: int = 8, seed: int = 0) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(seed)
    topics = [
        "the heat death of the universe",
        "Banach-Tarski paradox",
        "why inner ears sense rotation",
        "the illusion of explanatory depth",
        "neutron star crusts",
        "language death and star names",
        "data science communities",
        "stress and motivated reasoning",
        "Jevons paradox",
        "Stern-Gerlach experiment",
        "allopatric speciation",
        "type-token curves in small corpora",
    ]
    rows = []
    for p in range(n_prompts):
        topic = topics[p % len(topics)]
        pid = f"p{p:03d}"
        for model in LABELS:
            body = _synthetic_passage(model, topic, rng)
            rows.append(
                {
                    "id": f"{pid}-{model}",
                    "prompt_id": pid,
                    "text": body,
                    "model": model,
                }
            )
    train = pd.DataFrame(rows)
    test = train.sample(n=min(12, len(train)), random_state=seed).copy()
    test = test[["id", "text"]].reset_index(drop=True)
    return train, test


def _synthetic_passage(model: str, topic: str, rng: np.random.Generator) -> str:
    family = FAMILY_OF[model]
    n_words = int(rng.integers(45, 90))
    base = (
        f"A careful account of {topic} begins with the parts that actually do work, "
        f"not with a slogan. The mechanism is local, the consequences are global, "
        f"and the usual shortcuts hide the joints where the argument can break."
    )
    extra = []
    if family == "anthropic":
        extra.append("The claim is modest — the supporting structure is not.")
        extra.append("What remains is a straight path from evidence to consequence.")
    elif family == "google":
        extra.append("The tapestry of causes is more intricate than a single lever.")
        extra.append("In this landscape the key constraint is geometric, not verbal.")
    elif family == "openai":
        extra.append("That is, the same pattern shows up in small cases and in large ones.")
        extra.append("For instance, the counting argument and the geometric argument agree.")
    elif family == "qwen":
        extra.append("In other words, the bound is tight, and the examples are not decorative.")
    elif family == "zai":
        extra.append("The remaining uncertainty is practical — not conceptual.")
    else:
        extra.append("Hence the useful move is to keep the invariant in view.")

    if model.startswith("gpt-"):
        extra.append("Numbers, names, and commas do real work here, 1922, 5, and 18 alike.")
        text = base + " " + " ".join(extra)
        text = text.replace("'", "’")
        if "gpt-5.5" in model:
            text = text.replace(" — ", ", and ")
    elif model.startswith("gemini-"):
        text = (base + " " + " ".join(extra)).replace("—", ",").replace("’", "'")
        text = text.replace("n't", "not")
    elif model.startswith("claude-"):
        text = base + " " + " ".join(extra)
        text = text.replace("“", '"').replace("”", '"').replace("’", "'")
    else:
        text = base + " " + " ".join(extra)

    words = text.split()
    if len(words) < n_words:
        words = words + (["the", "same", "structure", "repeats"] * ((n_words - len(words)) // 4 + 1))
    text = " ".join(words[: max(n_words, 40)])
    if not text.endswith("."):
        text += "."
    return text


In [ ]:
"""Public extra-passage discovery and host-style cleaning."""

from __future__ import annotations

import re
from pathlib import Path
from typing import Sequence

import pandas as pd


SKIP_SUFFIXES = {
    ".safetensors",
    ".bin",
    ".pt",
    ".pth",
    ".ckpt",
    ".onnx",
    ".npz",
    ".gguf",
    ".msgpack",
}
SKIP_DIR_MARKERS = (
    "lora8b",
    "lora14b",
    "/adapter/",
    "\\adapter\\",
    "tokenizer/",
)
MD_RE = re.compile(r"```|^\s{0,3}#{1,6}\s|^\s*[-*]\s", re.M)
WORD_RE = re.compile(r"\S+")


def _is_weight_or_adapter(path: Path) -> bool:
    if path.suffix.lower() in SKIP_SUFFIXES:
        return True
    low = str(path).lower()
    return any(m in low for m in SKIP_DIR_MARKERS)


def discover_extra_files(roots: Sequence[str | Path]) -> list[Path]:
    skip_names = {
        "train.csv",
        "test.csv",
        "sample_submission.parquet",
        "sample_submission.csv",
        "synth_train.csv",
        "synth_test.csv",
    }
    found: list[Path] = []
    for root in roots:
        root_path = Path(root)
        if not root_path.exists():
            continue
        paths = [root_path] if root_path.is_file() else list(root_path.rglob("*"))
        for path in paths:
            if not path.is_file() or path.suffix.lower() not in {".csv", ".parquet"}:
                continue
            if path.name in skip_names or _is_weight_or_adapter(path):
                continue
            try:
                if path.stat().st_size > 120_000_000:
                    continue
            except OSError:
                continue
            found.append(path)
    return found


def illegal_marks_from_train(train: pd.DataFrame) -> dict[str, set[str]]:
    illegal: dict[str, set[str]] = {"emdash": set(), "curly": set()}
    if train is None or train.empty:
        return illegal
    for model, g in train.groupby("model"):
        if not g["text"].str.contains("—", regex=False).any():
            illegal["emdash"].add(str(model))
        if not g["text"].str.contains(r"[“”‘’]", regex=True).any():
            illegal["curly"].add(str(model))
    return illegal


def clean_extra(df: pd.DataFrame, train: pd.DataFrame | None = None) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame(columns=["id", "prompt_id", "text", "model"])
    out = df.copy()
    out = out[out["model"].isin(LABELS)]
    words = out["text"].map(lambda t: len(WORD_RE.findall(str(t))))
    out = out[(words >= 30) & (words <= 300)]
    out = out[~out["text"].map(lambda t: bool(MD_RE.search(str(t))))]
    illegal = illegal_marks_from_train(train) if train is not None else {"emdash": set(), "curly": set()}
    if illegal["emdash"]:
        drop = out["model"].isin(illegal["emdash"]) & out["text"].str.contains("—", regex=False)
        out = out[~drop]
    if illegal["curly"]:
        drop = out["model"].isin(illegal["curly"]) & out["text"].str.contains(r"[“”‘’]", regex=True)
        out = out[~drop]
    out = out.drop_duplicates(subset=["text"])
    return out.reset_index(drop=True)


def subsample_extra(df: pd.DataFrame, max_rows: int, seed: int = 0) -> pd.DataFrame:
    if df is None or len(df) <= max_rows:
        return df.reset_index(drop=True) if df is not None else df
    parts = []
    n_lab = df["model"].nunique() or 1
    per = max(max_rows // n_lab, 1)
    rng = seed
    for _, g in df.groupby("model"):
        take = min(len(g), per)
        parts.append(g.sample(n=take, random_state=rng) if take < len(g) else g)
    out = pd.concat(parts, ignore_index=True)
    if len(out) > max_rows:
        out = out.sample(n=max_rows, random_state=seed)
    return out.reset_index(drop=True)


def load_and_clean_extra(
    roots: Sequence[str | Path],
    train: pd.DataFrame,
    test_ids: Sequence[str] | None = None,
    max_rows: int = 16_000,
) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    train_text = set(train["text"].astype(str)) if train is not None else set()
    banned_ids = set(map(str, test_ids or []))
    for path in discover_extra_files(roots):
        try:
            raw = normalize_frame(load_table(path))
        except Exception:
            continue
        if "model" not in raw.columns:
            continue
        frames.append(raw)
    if not frames:
        return pd.DataFrame(columns=["id", "prompt_id", "text", "model"])
    extra = pd.concat(frames, ignore_index=True)
    extra = extra[~extra["text"].isin(train_text)]
    if banned_ids:
        extra = extra[~extra["id"].isin(banned_ids)]
    extra = clean_extra(extra, train)
    extra = subsample_extra(extra, max_rows=max_rows)
    return extra


In [ ]:
"""Qwen LoRA classifier for Name That LLM (Kaggle GPU, internet off)."""

from __future__ import annotations

import json
from pathlib import Path
from typing import Sequence

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoConfig, AutoModel, AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig


try:
    from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
except Exception:  # pragma: no cover - imported on Kaggle GPU images
    LoraConfig = TaskType = get_peft_model = prepare_model_for_kbit_training = None


QWEN_HINTS = ("qwen3", "qwen2", "qwen")
MAX_LEN = 384


def find_base_model(roots: Sequence[str | Path] | None = None) -> Path:
    search = [Path(p) for p in (roots or [])]
    search.extend(
        [
            Path("/kaggle/input"),
            Path("models"),
        ]
    )
    ranked: list[tuple[int, Path]] = []
    seen = set()
    for root in search:
        if not root.exists():
            continue
        for cfg in root.rglob("config.json"):
            if "adapter" in str(cfg).lower():
                continue
            try:
                data = json.loads(cfg.read_text())
            except Exception:
                continue
            model_type = str(data.get("model_type", "")).lower()
            arch = " ".join(map(str, data.get("architectures", []))).lower()
            blob = model_type + " " + arch + " " + str(cfg).lower()
            if not any(h in blob for h in QWEN_HINTS):
                continue
            parent = cfg.parent.resolve()
            if parent in seen:
                continue
            seen.add(parent)
            hidden = int(data.get("hidden_size") or data.get("n_embd") or 0)
            ranked.append((hidden, parent))
    if not ranked:
        raise FileNotFoundError(
            "No Qwen base model found. Attach qwen-lm/qwen-3 Transformers 8b-base "
            "(Kaggle Models) with internet off."
        )
    ranked.sort(reverse=True)
    return ranked[0][1]


class PassageDataset(Dataset):
    def __init__(self, texts: Sequence[str], labels: Sequence[int] | None, tokenizer, max_len: int = MAX_LEN):
        self.texts = [str(t) for t in texts]
        self.labels = None if labels is None else [int(x) for x in labels]
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict:
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def _label_ids(y: Sequence[str]) -> list[int]:
    index = {lab: i for i, lab in enumerate(LABELS)}
    return [index[str(v)] for v in y]


def _prepare_tokenizer(model_path: Path):
    tok = AutoTokenizer.from_pretrained(str(model_path), trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    return tok


def _bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )


class _PooledHead(torch.nn.Module):
    def __init__(self, backbone, hidden: int, n_labels: int):
        super().__init__()
        self.backbone = backbone
        self.score = torch.nn.Linear(hidden, n_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).to(h.dtype)
        pooled = (h * mask).sum(1) / mask.sum(1).clamp(min=1e-6)
        logits = self.score(pooled.float())
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)
        return type("Out", (), {"loss": loss, "logits": logits})


def build_model(model_path: Path, n_labels: int = 18, use_4bit: bool = True):
    tok = _prepare_tokenizer(model_path)
    max_mem = {0: "13GiB", 1: "13GiB", "cpu": "24GiB"} if torch.cuda.device_count() > 1 else None
    load_kw = dict(torch_dtype=torch.float16, trust_remote_code=True, device_map="auto")
    if max_mem is not None:
        load_kw["max_memory"] = max_mem
    if use_4bit:
        load_kw["quantization_config"] = _bnb_config()
    if LoraConfig is None:
        raise RuntimeError("peft is required for LoRA training")
    targets = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            str(model_path),
            num_labels=n_labels,
            pad_token_id=tok.pad_token_id,
            **load_kw,
        )
        model.config.pad_token_id = tok.pad_token_id
        model.config.problem_type = "single_label_classification"
        if use_4bit and prepare_model_for_kbit_training is not None:
            model = prepare_model_for_kbit_training(model)
        lora = LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            bias="none",
            task_type=TaskType.SEQ_CLS,
            target_modules=targets,
            modules_to_save=["score", "classifier"],
        )
        model = get_peft_model(model, lora)
    except Exception as exc:
        print(f"SEQ_CLS path failed ({exc}); using mean-pool head", flush=True)
        backbone = AutoModel.from_pretrained(str(model_path), **load_kw)
        if use_4bit and prepare_model_for_kbit_training is not None:
            backbone = prepare_model_for_kbit_training(backbone)
        lora = LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            bias="none",
            task_type=TaskType.CAUSAL_LM,
            target_modules=targets,
        )
        backbone = get_peft_model(backbone, lora)
        hidden = int(AutoConfig.from_pretrained(str(model_path), trust_remote_code=True).hidden_size)
        model = _PooledHead(backbone, hidden, n_labels)
        if torch.cuda.is_available():
            model.score = model.score.to(next(backbone.parameters()).device)
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
        if hasattr(model, "enable_input_require_grads"):
            model.enable_input_require_grads()
    if hasattr(model, "print_trainable_parameters"):
        model.print_trainable_parameters()
    return model, tok, use_4bit


def _class_weights(y_ids: Sequence[int], n_labels: int = 18) -> torch.Tensor:
    counts = np.bincount(list(y_ids), minlength=n_labels).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    w = counts.sum() / (n_labels * counts)
    return torch.tensor(w, dtype=torch.float32)


def _move(batch: dict, device: torch.device) -> dict:
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}


def _model_device(model) -> torch.device:
    if hasattr(model, "backbone"):
        return next(model.backbone.parameters()).device
    return next(model.parameters()).device


def train_epochs(
    model,
    tok,
    texts: Sequence[str],
    labels: Sequence[str],
    epochs: int,
    lr: float,
    batch_size: int = 1,
    accum: int = 16,
    class_weight: torch.Tensor | None = None,
) -> None:
    y_ids = _label_ids(labels)
    ds = PassageDataset(texts, y_ids, tok)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    device = _model_device(model)
    if class_weight is not None:
        class_weight = class_weight.to(device)
    model.train()
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    steps = 0
    opt.zero_grad(set_to_none=True)
    for epoch in range(epochs):
        running = 0.0
        n = 0
        for batch in loader:
            batch = _move(batch, device)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available(), dtype=torch.float16):
                out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
                logits = out.logits
                loss = F.cross_entropy(logits, batch["labels"], weight=class_weight)
                loss = loss / accum
            scaler.scale(loss).backward()
            running += float(loss.item()) * accum
            n += 1
            steps += 1
            if steps % accum == 0:
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
        if steps % accum != 0:
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
        print(f"epoch {epoch + 1}/{epochs} mean_loss={running / max(n, 1):.4f}", flush=True)


@torch.no_grad()
def predict_proba(model, tok, texts: Sequence[str], batch_size: int = 4) -> np.ndarray:
    model.eval()
    device = _model_device(model)
    ds = PassageDataset(texts, None, tok)
    # length sort for fewer pads on the hidden rerun
    order = np.argsort([len(t.split()) for t in texts])
    inv = np.empty_like(order)
    inv[order] = np.arange(len(order))
    ordered = [texts[i] for i in order]
    ds.texts = [str(t) for t in ordered]
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    chunks = []
    for batch in loader:
        batch = _move(batch, device)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available(), dtype=torch.float16):
            logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
        chunks.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    probs = np.concatenate(chunks, axis=0)[inv]
    return probs


def two_stage_train(
    model_path: Path,
    train: pd.DataFrame,
    extra: pd.DataFrame | None = None,
    epochs_extra: int = 1,
    epochs_host: int = 2,
) -> tuple:
    model, tok, _ = build_model(model_path)
    y_host = train["model"].astype(str).tolist()
    weights = _class_weights(_label_ids(y_host))
    if extra is not None and len(extra):
        print(f"LoRA stage 1: extra n={len(extra)}", flush=True)
        train_epochs(
            model,
            tok,
            extra["text"].astype(str).tolist(),
            extra["model"].astype(str).tolist(),
            epochs=epochs_extra,
            lr=2.0e-4,
            class_weight=weights,
        )
    print(f"LoRA stage 2: host n={len(train)}", flush=True)
    train_epochs(
        model,
        tok,
        train["text"].astype(str).tolist(),
        y_host,
        epochs=epochs_host if extra is not None and len(extra) else max(epochs_host, 3),
        lr=8.0e-5,
        class_weight=weights,
    )
    return model, tok


def blend_and_penalize(
    p_lora: np.ndarray,
    p_tfidf: np.ndarray | None,
    texts: Sequence[str],
    train_texts: Sequence[str],
    train_y: Sequence[str],
    lora_weight: float = 0.78,
    rebalance: bool = True,
) -> np.ndarray:
    if p_tfidf is None:
        probs = np.asarray(p_lora, dtype=np.float64)
    else:
        w = lora_weight
        probs = w * np.asarray(p_lora, dtype=np.float64) + (1.0 - w) * np.asarray(p_tfidf, dtype=np.float64)
    illegal = fit_mark_illegal(train_texts, train_y)
    probs = apply_mark_penalties(probs, texts, list(LABELS), illegal, factor=0.18)
    if rebalance:
        probs = rebalance_probs(probs)
    return probs


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

print("Name That LLM — Qwen LoRA + TF-IDF blend")

def _pick(paths):
    for p in paths:
        if Path(p).exists():
            return Path(p)
    return None

train_path = _pick([
    "/kaggle/input/competitions/name-that-llm/train.csv",
    "/kaggle/input/name-that-llm/train.csv",
    "data/train.csv",
])
test_path = _pick([
    "/kaggle/input/competitions/name-that-llm/test.csv",
    "/kaggle/input/name-that-llm/test.csv",
    "data/test.csv",
])
if train_path is None or test_path is None:
    raise FileNotFoundError("Attach the competition dataset (train.csv / test.csv).")

train = normalize_frame(load_table(train_path))
test = normalize_frame(load_table(test_path))
extra = load_and_clean_extra(
    ["/kaggle/input", Path("data/external")],
    train,
    test_ids=test["id"].astype(str),
    max_rows=16_000,
)
print(f"train={len(train)} test={len(test)} extra={len(extra)}")
if extra.empty:
    print("WARNING: no extra public passages attached. Bronze usually needs a public pool.")

texts = train["text"].astype(str).tolist()
labels = train["model"].astype(str).tolist()
weights = [1.0] * len(texts)
if len(extra):
    texts.extend(extra["text"].astype(str).tolist())
    labels.extend(extra["model"].astype(str).tolist())
    weights.extend([0.55] * len(extra))

print("Fitting TF-IDF / logistic regression", flush=True)
tfidf = NTLClassifier(extra_weight=0.55, rebalance=False)
tfidf.fit(texts, labels, sample_weight=np.asarray(weights, dtype=float))
p_tfidf = tfidf.predict_proba(test["text"].astype(str).tolist())
# align to LABELS
col = {str(c): i for i, c in enumerate(tfidf.classes_)}
p_tfidf_aligned = np.zeros((len(test), len(LABELS)))
for j, lab in enumerate(LABELS):
    if lab in col:
        p_tfidf_aligned[:, j] = p_tfidf[:, col[lab]]

print("Loading Qwen base + LoRA", flush=True)
base = find_base_model(["/kaggle/input", "models"])
print("base model:", base, flush=True)
model, tok = two_stage_train(base, train, extra if len(extra) else None)
p_lora = predict_proba(model, tok, test["text"].astype(str).tolist(), batch_size=4)
# map LoRA columns (LABELS order)
assert p_lora.shape[1] == len(LABELS)

probs = blend_and_penalize(
    p_lora,
    p_tfidf_aligned,
    test["text"].astype(str).tolist(),
    train["text"].astype(str).tolist(),
    train["model"].astype(str).tolist(),
    lora_weight=0.78,
    rebalance=True,
)
pred = [LABELS[i] for i in probs.argmax(axis=1)]
out_path = Path("submission.parquet")
write_submission(test["id"].astype(str), pred, out_path)
check = pd.read_parquet(out_path)
assert list(check.columns) == ["id", "model"]
assert check["id"].nunique() == len(test)
assert set(check["model"]).issubset(set(LABELS))
print(check.head())
print("wrote", out_path.resolve(), "rows", len(check), "extra", len(extra))
print(json.dumps({"n_train": len(train), "n_extra": int(len(extra)), "n_pred": len(check)}, indent=2))
